In [ ]:
# SPDX-FileCopyrightText: 2026 Mario Gemoll
# SPDX-License-Identifier: 0BSD

import os
import subprocess


def is_correct_repo() -> bool:
    try:
        result = subprocess.run(
            ["git", "remote", "get-url", "origin"],
            capture_output=True,
            text=True,
            check=True,
        )
        remote_url = result.stdout.strip()
        return remote_url in [
            "https://github.com/mariogemoll/reinforcement-learning.git",
            "git@github.com:mariogemoll/reinforcement-learning.git",
        ]
    except (subprocess.CalledProcessError, FileNotFoundError):
        return False


if not is_correct_repo():
    !git clone https://github.com/mariogemoll/reinforcement-learning.git

if not os.getcwd().endswith("reinforcement-learning/py"):
    %cd reinforcement-learning/py


In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd() / "src"))


In [ ]:
%pip install -q gymnax anywidget

In [ ]:
import base64
from pathlib import Path

from IPython.display import display

from rl.core.dqn import fresh_params, run_config, make_model
from rl.minatar.visualizations import MinAtarBreakoutVisualization
from rl.core.util import eval_dqn_max_score, plot_dqn_metrics, as_f32, write_safetensors


In [ ]:
env_name = "Breakout-MinAtar"
cfg = {
    "env_name": env_name,
    "hidden_dim": 256,
    "num_layers": 1,
    "lr": 1.6e-3,
    "decay_dur": 200_000,
    "batch_size": 128,
    "buf_cap": 50_000,
    "learn_start": 2_000,
    "upd_every": 1000,
}

total_steps = 2_000_000

In [ ]:
init_params = fresh_params(
    env_name=cfg["env_name"],
    hidden_dim=cfg["hidden_dim"],
    num_layers=cfg["num_layers"],
)
ep_rets, losses, best_params = run_config(cfg, total_steps, init_params)
plot_dqn_metrics(losses, ep_rets)


In [ ]:
print(
    eval_dqn_max_score(
        best_params,
        env_name=cfg["env_name"],
        hidden_dim=cfg["hidden_dim"],
        num_layers=cfg["num_layers"]
    )
)

In [ ]:
# Export trained model weights in safetensors format.
model = make_model(
    best_params,
    env_name=cfg["env_name"],
    hidden_dim=cfg["hidden_dim"],
    num_layers=cfg["num_layers"],
)
tensors = {
    "dense1.weight": as_f32(model.layer_0.kernel.value),
    "dense1.bias": as_f32(model.layer_0.bias.value),
}
if cfg["num_layers"] >= 2:
    tensors["dense2.weight"] = as_f32(model.layer_1.kernel.value)
    tensors["dense2.bias"] = as_f32(model.layer_1.bias.value)
out_layer = getattr(model, f"layer_{model.num_layers - 1}")
tensors["dense_out.weight"] = as_f32(out_layer.kernel.value)
tensors["dense_out.bias"] = as_f32(out_layer.bias.value)
out_path = Path("dqn-minatar-breakout-weights.safetensors")
write_safetensors(out_path, tensors)
print(f"Wrote {out_path} ({out_path.stat().st_size:,} bytes)")


In [ ]:
%%bash
# Build MinAtar Breakout visualization
set -euo pipefail

cd ../ts

if command -v pnpm >/dev/null 2>&1; then
  echo "Using package manager: pnpm"
  pnpm i
  pnpm run build:anywidget-minatar-breakout
else
  echo "Using package manager: npm"
  npm install
  npm run build:anywidget-minatar-breakout
fi

In [ ]:
weights_bytes = Path("dqn-minatar-breakout-weights.safetensors").read_bytes()
weights_b64 = base64.b64encode(weights_bytes).decode("ascii")

display(MinAtarBreakoutVisualization(weights_base64=weights_b64))
